# Computer Vision Techniques Demo

This notebook demonstrates various computer vision techniques including:
- Image Filtering (Grayscale, Gaussian, Bilateral)
- Edge Detection (Sobel, Canny)
- Object Detection (Hough Transform, Template Matching)
- Custom Object Tracking with Ellipse Fitting

In [1]:
from typing import Callable
import cv2
import time
import numpy as np

def std_norm(img: cv2.Mat, nstd: float = 2):
    """Normalize image using mean and standard deviation."""
    mean, std = cv2.meanStdDev(img)
    normalized = (img - mean) / (nstd * std) * 255 + 128
    normalized = np.clip(normalized, 0, 255).astype(np.uint8)
    return normalized

FrameMapper = Callable[[cv2.Mat], cv2.Mat]

def draw_frame_corners(img: cv2.Mat, thickness=5, length=40):
    """Draw decorative corner markers on the image."""
    h, w = img.shape[:2]
    img[0:thickness, 0:length] = 0
    img[0:length, 0:thickness] = 0
    img[0:thickness, w-length:w] = 0
    img[0:length, w-thickness:w] = 0
    img[h-thickness:h, 0:length] = 0
    img[h-length:h, 0:thickness] = 0
    img[h-thickness:h, w-length:w] = 0
    img[h-length:h, w-thickness:w] = 0

def process_frames(
        cap: cv2.VideoCapture,
        mapper: FrameMapper = None,
        start_sec: float = None,
        end_sec: float = None,
        out: cv2.VideoWriter = None
) -> None:
    """Process video frames in a time range with a mapper function."""
    mapper = (lambda x: x) if mapper is None else mapper
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    start_frame = 0 if start_sec is None else int(start_sec * fps)
    end_frame = int(end_sec * fps) if end_sec is not None else int(1e18)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    start_time = time.time()

    current_frame = start_frame
    while current_frame < end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        processed_frame = mapper(frame)
        cv2.imshow('Processed Frame', processed_frame)
        if out is not None:
            out.write(processed_frame)
        else:
            frame_end_time = start_time + (current_frame - start_frame + 1) / fps
            delta_time = frame_end_time - time.time()
            time_to_wait = max(0, delta_time)
            time.sleep(time_to_wait)

        cv2.waitKey(1)
        current_frame += 1

def putTextShadow(img, text, org, fontFace, fontScale, color, thickness, 
                  lineType=cv2.LINE_8, bottomLeftOrigin=False, 
                  shadowOffset=(1, 1), shadowColor=(100, 100, 100)):
    """Draw text with a shadow for better visibility."""
    shadow_org = (org[0] + shadowOffset[0], org[1] + shadowOffset[1])
    cv2.putText(img, text, shadow_org, fontFace, fontScale, shadowColor, 
                thickness+1, lineType, bottomLeftOrigin)
    cv2.putText(img, text, org, fontFace, fontScale, color, 
                thickness, lineType, bottomLeftOrigin)

def bind(m1, m2):
    """Compose two mappers: apply m1 first, then m2."""
    return lambda img: m2(m1(img))

In [2]:
# Initialize video capture and writer
cap = cv2.VideoCapture('input_video.mp4')

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_demo.mp4', fourcc, fps, (w, h))

mb = 60  # Bottom margin for text

## 1. Grayscale Filter Demo

In [3]:
def id_mapper(img: cv2.Mat):
    """Identity mapper with 'OFF' label."""
    putTextShadow(img, "Grayscale Filter: OFF", (mb, h-mb+10), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 230, 150), 2, cv2.LINE_AA)
    return img

def gray_scale3(img: cv2.Mat):
    """Apply grayscale to center region."""
    m = 64
    bw = w - 2*m
    img1 = img[m:h-mb-m, m:m+bw]
    img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
    draw_frame_corners(img1)
    img[m:h-mb-m, m:m+bw] = img1
    putTextShadow(img, "Grayscale Filter: ON", (mb, h-mb+10), cv2.FONT_HERSHEY_COMPLEX, 1, (240, 240, 240), 2, cv2.LINE_AA)
    return img

process_frames(cap, id_mapper, 0, 1, out)
process_frames(cap, gray_scale3, 1, 2, out)
process_frames(cap, id_mapper, 2, 3, out)
process_frames(cap, gray_scale3, 3, 4, out)

## 2. Gaussian vs Bilateral Filter Comparison

In [4]:
def get_blur_mapper(gsize, gsigma, blsize, blsigmaC, blsigmaS):
    """Create a mapper comparing Gaussian and Bilateral filters."""
    m = 64
    bw = (w-3*m)//2
    def mapper(img: cv2.Mat):
        # Left panel: Gaussian blur
        img1 = img[m:h-mb-m, m:m+bw]
        cv2.GaussianBlur(img1, gsize, gsigma, img1)
        draw_frame_corners(img1)
        
        # Right panel: Bilateral filter
        img2 = img[m:h-mb-m, m+bw+m:m+bw+m+bw]
        img22 = cv2.bilateralFilter(img2, blsize, blsigmaC, blsigmaS)
        draw_frame_corners(img22)
        img[m:h-mb-m, m+bw+m:m+bw+m+bw] = img22

        # Labels
        putTextShadow(img, "Gaussian Filter:", (m, h-mb-30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"kSize={gsize}", (m+20, h-mb), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"Sigma={gsigma}", (m+20, h-mb+30), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)

        putTextShadow(img, "Bilateral Filter:", (m+bw+m, h-mb-30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"kSize={blsize}", (m+bw+m+20, h-mb), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"Sigma(Color, Space)=({blsigmaC}, {blsigmaS})", (m+bw+m+20, h-mb+30), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)

        return img
    return mapper

for t in range(4, 12):
    d = (t - 4)*2 + 1
    mapper = get_blur_mapper((d, d), 3, d, 75, 75)
    process_frames(cap, mapper, t, t+1, out)

## 3. Color-Based Segmentation

In [5]:
def grab_mapper00(img: cv2.Mat):
    """Show original with title."""
    putTextShadow(img, "Grab:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    return img

def grab_mapper0(img: cv2.Mat):
    """Step 1: Color distance visualization."""
    color = np.array([[[216, 245, 114]]])
    cdist = np.linalg.norm(color - img, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    putTextShadow(img, "Grab:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img[45:65, 810:830] = color
    return img

def grab_mapper1(img: cv2.Mat):
    """Step 2: Thresholding."""
    img = grab_mapper0(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, img = cv2.threshold(img, 50, 255, cv2.THRESH_BINARY_INV)

    putTextShadow(img, "Grab:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img[45:65, 810:830] = np.array([[[216, 245, 114]]])
    putTextShadow(img, f"2. Threshold (Inverse): 50", (560, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    return img

def grab_mapper2(img: cv2.Mat):
    """Step 3: Morphological operations."""
    img = grab_mapper1(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img0 = img
    k = 60
    img = cv2.copyMakeBorder(img, k, k, k, k, cv2.BORDER_CONSTANT, value=0)
    kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel1)
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    img = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel2)
    img = img[k:-k, k:-k]
    img1 = img
    
    # Merge visualization
    img = np.zeros((*img0.shape, 3), dtype=np.uint8)
    img[(img0 == 255) & (img1 == 255)] = [255, 255, 255]
    img[(img0 == 0) & (img1 == 255)] = [0, 255, 0]
    img[(img0 == 255) & (img1 == 0)] = [0, 0, 255]

    putTextShadow(img, "Grab:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img[45:65, 810:830] = np.array([[[216, 245, 114]]])
    putTextShadow(img, f"2. Threshold (Inverse): 50", (560, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"3. Morphology Op: ", (560, 120), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"+", (815, 120), cv2.FONT_HERSHEY_COMPLEX, 0.8, (0, 255, 0), 3, cv2.LINE_AA)
    putTextShadow(img, f"-", (845, 120), cv2.FONT_HERSHEY_COMPLEX, 0.8, (0, 0, 255), 3, cv2.LINE_AA)
    return img

process_frames(cap, grab_mapper00, 12, 14, out)
process_frames(cap, grab_mapper0, 14, 16, out)
process_frames(cap, grab_mapper1, 16, 18, out)
process_frames(cap, grab_mapper2, 18, 20, out)

## 4. Sobel Edge Detection

In [6]:
def get_sobel(ksize: int):
    """Create Sobel edge detection mapper."""
    m = 64
    bw = (w-3*m)//2
    def mapper(img: cv2.Mat):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        imgx = img[m:h-mb-m, m:m+bw]
        imgy = img[m:h-mb-m, m+bw+m:m+bw+m+bw]
        Gx = cv2.Sobel(imgx, cv2.CV_64F, 1, 0, ksize=ksize)
        Gy = cv2.Sobel(imgy, cv2.CV_64F, 0, 1, ksize=ksize)
        Gx, Gy = np.abs(Gx), np.abs(Gy)
        Gx, Gy = std_norm(Gx), std_norm(Gy)
        img[m:h-mb-m, m:m+bw] = Gx
        img[m:h-mb-m, m+bw+m:m+bw+m+bw] = Gy
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        putTextShadow(img, "Sobel X:", (m, h-mb-30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"kSize={ksize}", (m+20, h-mb), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"* Std Dev Normalized", (m-20, h-mb+40), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)

        putTextShadow(img, "Sobel Y:", (m+bw+m, h-mb-30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"kSize={ksize}", (m+bw+m+20, h-mb), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
        return img
    return mapper

for t in range(20, 25):
    ksize = (t - 20)*4 + 1
    sobel = get_sobel(ksize)
    process_frames(cap, sobel, t, t+1, out)

## 5. Hough Circle Detection

In [7]:
def hough0(img: cv2.Mat):
    putTextShadow(img, "Object Detection:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    return img

def hough1(img: cv2.Mat):
    img = hough0(img)
    color = np.array([[[216, 245, 114]]])
    cdist = np.linalg.norm(color - img, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    putTextShadow(img, "Object Detection:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img[45:65, 810:830] = color
    return img

def hough2(img: cv2.Mat):
    img = hough1(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.GaussianBlur(img, (7, 7), 5)

    putTextShadow(img, "Object Detection:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img[45:65, 810:830] = np.array([[[216, 245, 114]]])
    putTextShadow(img, f"2. Gaussian:", (560, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"     kSize=(7,7) Sigma=5", (560, 120), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
    return img

def hough3(img: cv2.Mat):
    img = hough1(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.GaussianBlur(img, (7, 7), 5)
    img = cv2.Canny(img, 220, 200)

    putTextShadow(img, "Object Detection:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    img[45:65, 810:830] = np.array([[[216, 245, 114]]])
    putTextShadow(img, f"2. Gaussian:", (560, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"     kSize=(7,7) Sigma=5", (560, 120), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"3.a Canny:", (560, 150), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    putTextShadow(img, f"      threshold=(220, 200)", (560, 180), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
    return img

def get_hough(param2, minDist):
    def hough(img: cv2.Mat):
        color = np.array([[[216, 245, 114]]])
        cdist = np.linalg.norm(color - img, axis=2)
        img1 = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        img1 = cv2.GaussianBlur(img1, (9, 9), 9)
        circles = cv2.HoughCircles(img1, cv2.HOUGH_GRADIENT, dp=16, minDist=minDist, param1=220, param2=param2, minRadius=50, maxRadius=200)
        
        if circles is not None:
            circles = np.uint16(np.around(circles))
            for (x, y, r) in circles[0, :]:
                cv2.circle(img, (x, y), r, (0, 200, 0), 2)
                cv2.circle(img, (x, y), 2, (0, 0, 200), 3)

        putTextShadow(img, "Object Detection:", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"1. Color distance: ", (560, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        img[45:65, 810:830] = np.array([[[216, 245, 114]]])
        putTextShadow(img, f"2. Gaussian:", (560, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     kSize=(9,9) Sigma=9", (560, 120), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"3.b HoughCircles:", (560, 150), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     * Min Dist: {minDist:.0f}, param2={param2:.0f}", (560, 180), cv2.FONT_HERSHEY_COMPLEX, 0.6, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     * Radius: [50, 200]", (560, 210), cv2.FONT_HERSHEY_COMPLEX, 0.5, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     * Canny threshold (high) = 220", (560, 230), cv2.FONT_HERSHEY_COMPLEX, 0.5, (240, 240, 240), 1, cv2.LINE_AA)
        return img
    return hough

process_frames(cap, hough0, 25, 26, out)
process_frames(cap, hough1, 26, 27, out)
process_frames(cap, hough2, 27, 28, out)
process_frames(cap, hough3, 28, 29, out)

for t in np.arange(29, 32, 1/fps):
    r = (t-29) / 3.0
    minDist = (1-r)*50 + r*501
    process_frames(cap, get_hough(20, minDist), t, t+1/fps, out)

for t in np.arange(32, 35, 1/fps):
    r = (t-32) / 3.0
    p2 = (1-r)*50 + r*251
    process_frames(cap, get_hough(p2, 20), t, t+1/fps, out)

## 6. Template Matching

In [8]:
template = cv2.imread("tracking_template.png")

def template_matching_mapper0(img: cv2.Mat):
    """Template matching with bounding box."""
    result = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)
    result = cv2.copyMakeBorder(result, 16, 15, 16, 15, cv2.BORDER_CONSTANT, value=0)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
    cx, cy = max_loc
    img = cv2.rectangle(img, (cx-16, cy-16), (cx+16, cy+16), (255, 0, 0), 2)

    putTextShadow(img, "Template Matching: ", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
    img[10:42, 810:842] = template
    return img

def template_matching_mapper1(img: cv2.Mat):
    """Template matching score visualization."""
    result = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)
    result = cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    result = cv2.copyMakeBorder(result, 16, 16, 16, 16, cv2.BORDER_CONSTANT, value=0)

    img = cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)
    putTextShadow(img, "Template Matching: ", (530, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (255, 255, 255), 1, cv2.LINE_AA)
    img[10:42, 810:842] = template
    return img

process_frames(cap, template_matching_mapper0, 35, 37, out)
process_frames(cap, template_matching_mapper1, 37, 40, out)

## 7. Custom Object Tracking Pipeline

This section demonstrates a multi-step tracking pipeline:
1. Color distance calculation
2. Gaussian blur
3. Canny edge detection
4. Contour detection
5. Ellipse fitting
6. Trajectory smoothing and hue shifting

In [9]:
# Pre-compute ellipse trajectory for smoothing
def detect_ellipse(img: np.ndarray):
    """Detect ellipse in frame using color-based segmentation."""
    color = np.array([[[216, 245, 114]]], dtype=np.float32)
    cdist = np.linalg.norm(img.astype(np.float32) - color, axis=2)
    norm_img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    blurred = cv2.GaussianBlur(norm_img, (9, 9), 3)
    edges = cv2.Canny(blurred, 200, 100)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_contours = [c for c in contours if len(c) > 500]
    
    if valid_contours:
        contour = max(valid_contours, key=len)
        ellipse = cv2.fitEllipse(contour)
        return ellipse
    return None

def capture_ellipse(cap: cv2.VideoCapture, start_sec: float, end_sec: float):
    """Capture ellipse detections from a video segment."""
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(start_sec * fps)
    end_frame = int(end_sec * fps)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ellipses = []
    frame_index = start_frame
    while frame_index < end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        ellipse = detect_ellipse(frame)
        ellipses.append(ellipse)
        frame_index += 1
    return ellipses

def interpolate_ellipse(ellipses):
    """Interpolate missing ellipse detections."""
    ellipses = ellipses[:]
    for i in range(len(ellipses)):
        if ellipses[i] is None:
            prev_index = next((j for j in range(i-1, -1, -1) if ellipses[j] is not None), None)
            next_index = next((k for k in range(i+1, len(ellipses)) if ellipses[k] is not None), None)
            
            if prev_index is not None and next_index is not None:
                frac = (i - prev_index) / (next_index - prev_index)
                center_prev, axes_prev, angle_prev = ellipses[prev_index]
                center_next, axes_next, angle_next = ellipses[next_index]
                
                center_interp = (center_prev[0] + frac * (center_next[0] - center_prev[0]),
                                 center_prev[1] + frac * (center_next[1] - center_prev[1]))
                axes_interp = (axes_prev[0] + frac * (axes_next[0] - axes_prev[0]),
                               axes_prev[1] + frac * (axes_next[1] - axes_prev[1]))
                angle_interp = angle_prev + frac * (angle_next - angle_prev)
                ellipses[i] = (center_interp, axes_interp, angle_interp)
            elif prev_index is not None:
                ellipses[i] = ellipses[prev_index]
            elif next_index is not None:
                ellipses[i] = ellipses[next_index]
    return ellipses

def smooth_ellipses(ellipses, kernel_size=5, sigma=2):
    """Smooth ellipse trajectory using Gaussian convolution."""
    if len(ellipses) == 0:
        return []

    centers = np.array([e[0] for e in ellipses])
    sizes = np.array([e[1] for e in ellipses])
    angles = np.array([e[2] for e in ellipses])

    kernel = cv2.getGaussianKernel(kernel_size, sigma).flatten()
    convolve = lambda data, k: np.convolve(data, k, mode='same')

    smoothed_centers = np.copy(centers)
    smoothed_centers[:, 0] = convolve(centers[:, 0], kernel)
    smoothed_centers[:, 1] = convolve(centers[:, 1], kernel)

    smoothed_sizes = np.copy(sizes)
    smoothed_sizes[:, 0] = convolve(sizes[:, 0], kernel)
    smoothed_sizes[:, 1] = convolve(sizes[:, 1], kernel)

    angles_rad = np.deg2rad(angles)
    sin_smoothed = convolve(np.sin(angles_rad), kernel)
    cos_smoothed = convolve(np.cos(angles_rad), kernel)
    smoothed_angles = np.rad2deg(np.arctan2(sin_smoothed, cos_smoothed))

    return [((smoothed_centers[i, 0], smoothed_centers[i, 1]),
             (smoothed_sizes[i, 0], smoothed_sizes[i, 1]),
             smoothed_angles[i])
            for i in range(len(ellipses))]

# Pre-compute trajectory
ellipsis = capture_ellipse(cap, 54, 60)
ellipsis = interpolate_ellipse(ellipsis)
ellipsis = smooth_ellipses(ellipsis)

In [10]:
def get_contour_color(contour):
    """Generate a color based on contour position."""
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        cx, cy = np.mean(contour[:, 0, :], axis=0).astype(int)
    hue = int((cx / w) * 179 + 100) % 180
    saturation = int((cy / h) * 255)
    value = 255
    hsv_color = np.uint8([[[hue, saturation, value]]])
    bgr_color = cv2.cvtColor(hsv_color, cv2.COLOR_HSV2BGR)[0][0].tolist()
    return bgr_color

# Step-by-step visual processing mappers
def mapper_4_1(img: np.ndarray):
    """Step 1: Color distance heatmap."""
    color = np.array([[[216, 245, 114]]], dtype=np.float32)
    cdist = np.linalg.norm(img.astype(np.float32) - color, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img

def mapper_4_2(img: cv2.Mat):
    """Step 2: Blurred heatmap."""
    img = mapper_4_1(img)
    img = cv2.GaussianBlur(img, (9, 9), 3)
    return img

def mapper_4_3(img: cv2.Mat):
    """Step 3: Edge detection on heatmap."""
    img = mapper_4_2(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.Canny(img, 200, 100)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img

def mapper_4_4(img: cv2.Mat):
    """Step 4: Draw all contours on original."""
    img0 = img
    color = np.array([[[216, 245, 114]]], dtype=np.float32)
    cdist = np.linalg.norm(img.astype(np.float32) - color, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = cv2.GaussianBlur(img, (9, 9), 3)
    img = cv2.Canny(img, 200, 100)
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        c = get_contour_color(contour)
        cv2.drawContours(img0, [contour], -1, c, 4)
    return img0

def mapper_4_5(img: cv2.Mat):
    """Step 5: Draw longest contour only."""
    img0 = img
    color = np.array([[[216, 245, 114]]], dtype=np.float32)
    cdist = np.linalg.norm(img.astype(np.float32) - color, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = cv2.GaussianBlur(img, (9, 9), 3)
    img = cv2.Canny(img, 200, 100)
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = [c for c in contours if len(c) > 500]
    if contours:
        contour = max(contours, key=len)
        c = get_contour_color(contour)
        cv2.drawContours(img0, [contour], -1, c, 4)
    return img0

def mapper_4_6(img: cv2.Mat):
    """Step 6: Fit and draw ellipse."""
    img0 = img
    color = np.array([[[216, 245, 114]]], dtype=np.float32)
    cdist = np.linalg.norm(img.astype(np.float32) - color, axis=2)
    img = cv2.normalize(cdist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = cv2.GaussianBlur(img, (9, 9), 3)
    img = cv2.Canny(img, 200, 100)
    contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = [c for c in contours if len(c) > 500]
    if contours:
        contour = max(contours, key=len)
        c = get_contour_color(contour)
        ellipse = cv2.fitEllipse(contour)
        cv2.ellipse(img0, ellipse, c, 4)
    return img0

# Smoothed tracking mappers
index = 0

def mapper_4_7(img: cv2.Mat):
    """Step 7: Draw smoothed ellipse."""
    global index
    ellipse = ellipsis[index]
    cv2.ellipse(img, ellipse, (255, 0, 255), 4)
    index += 1
    return img

def mapper_4_8(img: cv2.Mat):
    """Step 8: Hue shift effect inside ellipse."""
    global index
    ellipse = ellipsis[index]
    hue_shift = fps*180*index/(len(ellipsis) - 1)/6
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.ellipse(mask, ellipse, 255, -1)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h_channel = hsv[:, :, 0]
    indices = np.where(mask == 255)
    h_channel[indices] = (h_channel[indices].astype(np.int32) + hue_shift) % 180
    hsv[:, :, 0] = h_channel
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    index += 1
    return img

# Text overlay generator (applied AFTER visual processing)
def gText(ln):
    """Generate text overlay for step ln."""
    def mapper(img: cv2.Mat):
        i0 = 530
        i1 = 560
        i2 = 810

        putTextShadow(img, "Original:" if ln == 0 else "Process Steps", (i0, 30), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 0: return img
        putTextShadow(img, f"1. Color distance: ", (i1, 60), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        img[45:65, i2:i2+20] = np.array([[[216, 245, 114]]])
        if ln == 1: return img
        putTextShadow(img, f"2. Gaussian:", (i1, 90), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     kSize=(9,9) Sigma=3", (i1, 120), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 2: return img
        putTextShadow(img, f"3. Canny:", (i1, 150), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     threshold=(200, 100)", (i1, 180), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 3: return img
        putTextShadow(img, f"4. Find Contours:", (i1, 210), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 4: return img
        putTextShadow(img, f"     Find longest Contour", (i1, 240), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 5: return img
        putTextShadow(img, f"5. Fit Ellipse", (i1, 270), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 6: return img
        putTextShadow(img, f"     *Interpolate gaps", (i1, 300), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        putTextShadow(img, f"     *Smooth trajectory", (i1, 330), cv2.FONT_HERSHEY_COMPLEX, 0.7, (240, 240, 240), 1, cv2.LINE_AA)
        if ln == 7: return img
        putTextShadow(img, f"6. Shift Hue", (i1, 360), cv2.FONT_HERSHEY_COMPLEX, 0.8, (240, 240, 240), 1, cv2.LINE_AA)
        return img
    return mapper

# Run pipeline: bind(visual_mapper, text_mapper) applies visual first, then text
process_frames(cap, gText(0), 40, 42, out)
process_frames(cap, bind(mapper_4_1, gText(1)), 42, 44, out)
process_frames(cap, bind(mapper_4_2, gText(2)), 44, 46, out)
process_frames(cap, bind(mapper_4_3, gText(3)), 46, 48, out)
process_frames(cap, bind(mapper_4_4, gText(4)), 48, 50, out)
process_frames(cap, bind(mapper_4_5, gText(5)), 50, 52, out)
process_frames(cap, bind(mapper_4_6, gText(6)), 52, 54, out)
process_frames(cap, bind(mapper_4_7, gText(7)), 54, 57, out)
process_frames(cap, bind(mapper_4_8, gText(8)), 57, 60, out)

In [11]:
# Cleanup
out.release()
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
cv2.waitKey(1)
cv2.waitKey(1)
cv2.waitKey(1)
print("Processing complete. Output saved to output_demo.mp4")

Processing complete. Output saved to output_demo.mp4
